In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")

# Parse timestamps as UTC datetimes
for df, ts_col in [(df_metric, "timestamp"), (df_trace, "timestamp"), (df_log, "timestamp")]:
    df[ts_col] = pd.to_datetime(df[ts_col], unit="s", utc=True)

# Incident window (UTC)
start = pd.to_datetime("2024-01-16T02:43:30Z", utc=True)
end = pd.to_datetime("2024-01-16T03:13:30Z", utc=True)

# --- 1) metric.csv ---
# Global stats: count, p95, mean, max
metric_global_count = df_metric.groupby(["cmdb_id", "kpi_name"])["value"].count().rename("count")
metric_global_p95 = df_metric.groupby(["cmdb_id", "kpi_name"])["value"].quantile(0.95).rename("p95")
metric_global_mean = df_metric.groupby(["cmdb_id", "kpi_name"])["value"].mean().rename("mean")
metric_global_max = df_metric.groupby(["cmdb_id", "kpi_name"])["value"].max().rename("max")

metric_global = pd.concat([metric_global_count, metric_global_p95, metric_global_mean, metric_global_max], axis=1).reset_index()

# Windowed metrics
m_window = df_metric[(df_metric["timestamp"] >= start) & (df_metric["timestamp"] <= end)]
metric_win_grp = m_window.groupby(["cmdb_id", "kpi_name"])["value"].agg(
    count_in_window="count", max_in_window="max"
).reset_index()
# earliest timestamp in window
earliest = m_window.groupby(["cmdb_id", "kpi_name"])["timestamp"].min().rename("earliest_timestamp_in_window").reset_index()

metric_window = metric_win_grp.merge(earliest, on=["cmdb_id", "kpi_name"], how="left")

# Merge global and window (keep all global rows)
metric_summary = metric_global.merge(metric_window, on=["cmdb_id", "kpi_name"], how="left")
# Fill missing window counts with 0, keep earliest_timestamp_in_window as NaT if missing
metric_summary["count_in_window"] = metric_summary["count_in_window"].fillna(0).astype(int)
metric_summary["max_in_window"] = metric_summary["max_in_window"].where(metric_summary["max_in_window"].notna(), pd.NA)

# Order columns as requested
metric_summary = metric_summary[[
    "cmdb_id", "kpi_name", "count", "p95", "mean", "max",
    "count_in_window", "max_in_window", "earliest_timestamp_in_window"
]]

# Sort by count_in_window desc and limit to top 20
metric_summary = metric_summary.sort_values(by="count_in_window", ascending=False).head(20)

# --- 2) trace.csv ---
trace_global_count = df_trace.groupby(["cmdb_id", "trace_name"])["value"].count().rename("count")
trace_global_p95 = df_trace.groupby(["cmdb_id", "trace_name"])["value"].quantile(0.95).rename("p95")
trace_global = pd.concat([trace_global_count, trace_global_p95], axis=1).reset_index()

t_window = df_trace[(df_trace["timestamp"] >= start) & (df_trace["timestamp"] <= end)]
trace_win_grp = t_window.groupby(["cmdb_id", "trace_name"])["value"].agg(
    count_in_window="count", max_in_window="max"
).reset_index()
trace_earliest = t_window.groupby(["cmdb_id", "trace_name"])["timestamp"].min().rename("earliest_timestamp_in_window").reset_index()

trace_window = trace_win_grp.merge(trace_earliest, on=["cmdb_id", "trace_name"], how="left")
trace_summary = trace_global.merge(trace_window, on=["cmdb_id", "trace_name"], how="left")
trace_summary["count_in_window"] = trace_summary["count_in_window"].fillna(0).astype(int)
trace_summary["max_in_window"] = trace_summary["max_in_window"].where(trace_summary["max_in_window"].notna(), pd.NA)
trace_summary = trace_summary[[
    "cmdb_id", "trace_name", "count", "p95", "count_in_window", "max_in_window", "earliest_timestamp_in_window"
]]
trace_summary = trace_summary.sort_values(by="count_in_window", ascending=False).head(20)

# --- 3) log.csv ---
log_global_count = df_log.groupby(["cmdb_id", "log_name"])["value"].count().rename("count")
log_global_p95 = df_log.groupby(["cmdb_id", "log_name"])["value"].quantile(0.95).rename("p95")
log_global = pd.concat([log_global_count, log_global_p95], axis=1).reset_index()

l_window = df_log[(df_log["timestamp"] >= start) & (df_log["timestamp"] <= end)]
log_win_grp = l_window.groupby(["cmdb_id", "log_name"])["value"].agg(
    count_in_window="count", max_in_window="max"
).reset_index()
log_earliest = l_window.groupby(["cmdb_id", "log_name"])["timestamp"].min().rename("earliest_timestamp_in_window").reset_index()

log_window = log_win_grp.merge(log_earliest, on=["cmdb_id", "log_name"], how="left")
log_summary = log_global.merge(log_window, on=["cmdb_id", "log_name"], how="left")
log_summary["count_in_window"] = log_summary["count_in_window"].fillna(0).astype(int)
log_summary["max_in_window"] = log_summary["max_in_window"].where(log_summary["max_in_window"].notna(), pd.NA)
log_summary = log_summary[[
    "cmdb_id", "log_name", "count", "p95", "count_in_window", "max_in_window", "earliest_timestamp_in_window"
]]
log_summary = log_summary.sort_values(by="count_in_window", ascending=False).head(20)

# Display the three compact tables (top 20 rows each)
metric_summary, trace_summary, log_summary

```
Out[1]:
```
```python
summary = (
    "Summary of findings for incident window 2024-01-16T02:43:30Z to 2024-01-16T03:13:30Z:\n"
    "- The telemetry shows checkoutservice as the primary problem area. Its metric KPIs in the window are high:\n"
    "  * CPU: global P95 ~9.15 and max ~18.9 (elevated compared with other services).\n"
    "  * Latency: latency-90 P95 ~0.95s and max ~0.96s (high request latency).\n"
    "  * Memory: mem P95 and max are large (~1.34e8), consistent with resource pressure.\n"
    "- Trace data for checkoutservice also shows elevated durations and high row_counts during the window (all trace series have full coverage in-window and increased p95/max values).\n"
    "- Log data does not show increased error_count (error_count is 0), but row_count is present and large; redis logs have fewer points and appear later.\n"
    "Conclusion: checkoutservice is the likely root-cause component, with high CPU and increased request latency (and signs of memory/resource pressure). Recommend focusing debugging on checkoutservice CPU/memory usage, profiling slow traces, and checking contention or load spikes during the reported window."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id    kpi_name  count           p95          mean           max  count_in_window  max_in_window earliest_timestamp_in_window
0         adservice         cpu     25  6.917741e-01  6.091614e-01  6.954140e-01               25   6.954140e-01    2024-01-16 02:46:00+00:00
1         adservice      diskio     25  6.704367e+04  6.089405e+04  7.340708e+04               25   7.340708e+04    2024-01-16 02:46:00+00:00
2         adservice  latency-50     25  2.980589e-03  2.968148e-03  2.999944e-03               25   2.999944e-03    2024-01-16 02:46:00+00:00
3         adservice  latency-90     25  4.618295e-03  4.600857e-03  4.620090e-03               25   4.620090e-03    2024-01-16 02:46:00+00:00
4         adservice         mem     25  1.148021e+08  1.147015e+08  1.148735e+08               25   1.148735e+08    2024-01-16 02:46:00+00:00
5         adservice      socket     25  4.000000e+00  4.000000e+00  4.000000e+00               25   4.000000e+00    2024-01-16 02:46:00+00:00
6         adservice    workload     25  1.483900e+01  1.432835e+01  1.505650e+01               25   1.505650e+01    2024-01-16 02:46:00+00:00
7       cartservice         cpu     25  2.331531e+00  2.037090e+00  2.456028e+00               25   2.456028e+00    2024-01-16 02:46:00+00:00
8       cartservice  latency-50     25  3.251624e-03  3.155332e-03  3.281562e-03               25   3.281562e-03    2024-01-16 02:46:00+00:00
9       cartservice  latency-90     25  6.507447e-03  5.186267e-03  6.696999e-03               25   6.696999e-03    2024-01-16 02:46:00+00:00
10      cartservice         mem     25  4.379744e+07  4.372706e+07  4.381518e+07               25   4.381518e+07    2024-01-16 02:46:00+00:00
11      cartservice      socket     25  4.000000e+00  4.000000e+00  4.000000e+00               25   4.000000e+00    2024-01-16 02:46:00+00:00
12      cartservice    workload     25  2.509906e+01  2.425971e+01  2.529220e+01               25   2.529220e+01    2024-01-16 02:46:00+00:00
13  checkoutservice         cpu     25  1.892303e+01  9.147456e+00  1.898522e+01               25   1.898522e+01    2024-01-16 02:46:00+00:00
15  checkoutservice  latency-50     25  3.967172e-01  3.173305e-01  4.450801e-01               25   4.450801e-01    2024-01-16 02:46:00+00:00
16  checkoutservice  latency-90     25  9.482424e-01  7.549833e-01  9.612710e-01               25   9.612710e-01    2024-01-16 02:46:00+00:00
17  checkoutservice         mem     25  1.341407e+08  7.100389e+07  1.341704e+08               25   1.341704e+08    2024-01-16 02:46:00+00:00
18  checkoutservice      socket     25  1.200000e+01  1.048000e+01  1.200000e+01               25   1.200000e+01    2024-01-16 02:46:00+00:00
22  currencyservice  latency-50     25  6.303996e-02  6.062964e-02  6.332411e-02               25   6.332411e-02    2024-01-16 02:46:00+00:00
19  checkoutservice    workload     25  1.133403e+00  9.067382e-01  1.217633e+00               25   1.217633e+00    2024-01-16 02:46:00+00:00,             cmdb_id                                trace_name  count         p95  count_in_window  max_in_window earliest_timestamp_in_window
0   checkoutservice  trace.from_checkoutservice.duration_mean     25    0.011848               25       0.013868    2024-01-16 02:46:00+00:00
1   checkoutservice   trace.from_checkoutservice.duration_p95     25    0.084462               25       0.085441    2024-01-16 02:46:00+00:00
2   checkoutservice     trace.from_checkoutservice.error_rate     25    0.000000               25       0.000000    2024-01-16 02:46:00+00:00
3   checkoutservice      trace.from_checkoutservice.row_count     25  273.000000               25     281.000000    2024-01-16 02:46:00+00:00
4   checkoutservice  trace.from_frontendservice.duration_mean     25    0.434938               25       0.441613    2024-01-16 02:46:00+00:00
5   checkoutservice   trace.from_frontendservice.duration_p95     25    0.878288               25       0.908586    2024-01-16 02:46:00+00:00
6   checkoutservice     trace.from_frontendservice.error_rate     25    0.000000               25       0.000000    2024-01-16 02:46:00+00:00
7   checkoutservice      trace.from_frontendservice.row_count     25   32.000000               25      34.000000    2024-01-16 02:46:00+00:00
8   checkoutservice             trace.from_root.duration_mean     25    0.076945               25       0.080139    2024-01-16 02:46:00+00:00
9   checkoutservice              trace.from_root.duration_p95     25    0.196561               25       0.270098    2024-01-16 02:46:00+00:00
10  checkoutservice                trace.from_root.error_rate     25    0.000000               25       0.000000    2024-01-16 02:46:00+00:00
11  checkoutservice                 trace.from_root.row_count     25  134.600000               25     135.000000    2024-01-16 02:46:00+00:00
12  checkoutservice    trace.to_checkoutservice.duration_mean     25    0.011848               25       0.013868    2024-01-16 02:46:00+00:00
13  checkoutservice     trace.to_checkoutservice.duration_p95     25    0.084462               25       0.085441    2024-01-16 02:46:00+00:00
14  checkoutservice       trace.to_checkoutservice.error_rate     25    0.000000               25       0.000000    2024-01-16 02:46:00+00:00
15  checkoutservice        trace.to_checkoutservice.row_count     25  273.000000               25     281.000000    2024-01-16 02:46:00+00:00
16  checkoutservice    trace.to_currencyservice.duration_mean     25    0.003042               25       0.003492    2024-01-16 02:46:00+00:00
17  checkoutservice     trace.to_currencyservice.duration_p95     25    0.000443               25       0.000518    2024-01-16 02:46:00+00:00
18  checkoutservice       trace.to_currencyservice.error_rate     25    0.000000               25       0.000000    2024-01-16 02:46:00+00:00
19  checkoutservice        trace.to_currencyservice.row_count     25  110.600000               25     111.000000    2024-01-16 02:46:00+00:00,                   cmdb_id         log_name  count     p95  count_in_window  max_in_window earliest_timestamp_in_window
0               adservice  log.error_count     25     0.0               25              0    2024-01-16 02:46:00+00:00
1               adservice    log.row_count     25   446.0               25            447    2024-01-16 02:46:00+00:00
2             cartservice  log.error_count     25     0.0               25              0    2024-01-16 02:46:00+00:00
3             cartservice    log.row_count     25   765.8               25            789    2024-01-16 02:46:00+00:00
4         checkoutservice  log.error_count     25     0.0               25              0    2024-01-16 02:46:00+00:00
5         checkoutservice    log.row_count     25    97.4               25            102    2024-01-16 02:46:00+00:00
6         currencyservice  log.error_count     25     0.0               25              0    2024-01-16 02:46:00+00:00
7         currencyservice    log.row_count     25  2364.0               25           2454    2024-01-16 02:46:00+00:00
8            emailservice  log.error_count     25     0.0               25              0    2024-01-16 02:46:00+00:00
9            emailservice    log.row_count     25    32.8               25             34    2024-01-16 02:46:00+00:00
10               frontend  log.error_count     25     0.0               25              0    2024-01-16 02:46:00+00:00
11               frontend    log.row_count     25  2641.0               25           2689    2024-01-16 02:46:00+00:00
12         paymentservice  log.error_count     25     0.0               25              0    2024-01-16 02:46:00+00:00
13         paymentservice    log.row_count     25    65.6               25             68    2024-01-16 02:46:00+00:00
14  recommendationservice  log.error_count     25     0.0               25              0    2024-01-16 02:46:00+00:00
15  recommendationservice    log.row_count     25   561.8               25            575    2024-01-16 02:46:00+00:00
19        shippingservice    log.row_count     25   470.0               25            472    2024-01-16 02:46:00+00:00
18        shippingservice  log.error_count     25     0.0               25              0    2024-01-16 02:46:00+00:00
17                  redis    log.row_count      5     5.0                5              5    2024-01-16 02:49:00+00:00
16                  redis  log.error_count      5     0.0                5              0    2024-01-16 02:49:00+00:00)```
```

In [ ]:
# In[2]:

# Reuse existing df_metric, start, end if present; otherwise (safe) ensure they exist
# (They were created in the prior cell execution.)
checkout_df = df_metric[df_metric["cmdb_id"] == "checkoutservice"].copy()

# 1) global P95 per kpi_name using full series (before window filtering)
global_p95 = checkout_df.groupby("kpi_name")["value"].quantile(0.95).rename("global_p95").reset_index()

# 2) filter to incident window
w_df = checkout_df[(checkout_df["timestamp"] >= start) & (checkout_df["timestamp"] <= end)].copy()

# Prepare per-kpi summary rows
rows = []
kpis = global_p95["kpi_name"].unique()
p95_map = dict(zip(global_p95["kpi_name"], global_p95["global_p95"]))

for k in kpis:
    thr = p95_map[k]
    grp = w_df[w_df["kpi_name"] == k].sort_values("timestamp")
    count_in_window = int(grp.shape[0])
    max_in_window = grp["value"].max() if count_in_window > 0 else pd.NA

    # anomaly boolean series (value >= global_p95) evaluated on in-window points
    if count_in_window > 0:
        b = grp["value"] >= thr
        anomalies_count = int(b.sum())
        earliest_anomaly_timestamp = grp.loc[b, "timestamp"].min() if anomalies_count > 0 else pd.NaT

        # longest consecutive run of True in b
        if anomalies_count == 0:
            longest_run = 0
        else:
            # identify contiguous groups
            grp_bool = b.reset_index(drop=True)
            groups = (grp_bool != grp_bool.shift(1)).cumsum()
            run_lengths = grp_bool.groupby(groups).agg(lambda s: s.sum() if s.iloc[0] else 0)
            longest_run = int(run_lengths.max())
    else:
        anomalies_count = 0
        earliest_anomaly_timestamp = pd.NaT
        longest_run = 0

    rows.append({
        "kpi_name": k,
        "global_p95": thr,
        "count_in_window": count_in_window,
        "max_in_window": max_in_window,
        "anomalies_count": anomalies_count,
        "earliest_anomaly_timestamp": earliest_anomaly_timestamp,
        "longest_consecutive_anomaly_run": longest_run
    })

checkout_anomaly_summary = pd.DataFrame(rows)[[
    "kpi_name", "global_p95", "count_in_window", "max_in_window",
    "anomalies_count", "earliest_anomaly_timestamp", "longest_consecutive_anomaly_run"
]]

# Keep compact: one row per checkoutservice KPI (no further truncation needed)
checkout_anomaly_summary

```
Out[2]:
```
Summary (checkoutservice, incident window 2024-01-16T02:43:30Z–2024-01-16T03:13:30Z)

- Overall: checkoutservice shows multiple KPIs exceeding their global P95 during the window, but the most pronounced and sustained anomaly is in the "socket" metric.
- Key findings (per KPI):
  - socket: 12 anomalies in-window, earliest at 2024-01-16T02:59:00Z, longest consecutive run = 12 (i.e., sustained throughout that period).
  - cpu: 2 anomaly points, earliest 2024-01-16T03:02:00Z, isolated spikes (longest run = 1).
  - mem: 2 anomaly points, earliest 2024-01-16T03:02:00Z, isolated.
  - latency-90 and latency-50: 2 anomalies each (latency-90 earliest 02:46:00Z, latency-50 earliest 03:03:00Z), short spikes.
  - diskio and workload: small counts of anomalies (diskio had one anomaly at 03:07:00Z).
- Interpretation: The sustained socket anomalies strongly indicate a socket-related issue (e.g., increased socket errors, connection saturation, or connection pool exhaustion) as the primary failure mode. CPU/memory/latency show brief spikes and are likely secondary effects or correlated symptoms.
- Recommendation: Focus immediate investigation on socket/networking for checkoutservice — check connection pools, open file/socket limits, socket error counters, upstream timeouts and network connectivity; then profile CPU/memory and traces around the socket-anomaly period.

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name    global_p95  count_in_window  max_in_window  anomalies_count earliest_anomaly_timestamp  longest_consecutive_anomaly_run
0         cpu  1.892303e+01               25   1.898522e+01                2  2024-01-16 03:02:00+00:00                                1
1      diskio  1.062814e+10               13   1.067904e+10                1  2024-01-16 03:07:00+00:00                                1
2  latency-50  3.967172e-01               25   4.450801e-01                2  2024-01-16 03:03:00+00:00                                1
3  latency-90  9.482424e-01               25   9.612710e-01                2  2024-01-16 02:46:00+00:00                                1
4         mem  1.341407e+08               25   1.341704e+08                2  2024-01-16 03:02:00+00:00                                1
5      socket  1.200000e+01               25   1.200000e+01               12  2024-01-16 02:59:00+00:00                               12
6    workload  1.133403e+00               25   1.217633e+00                2  2024-01-16 02:53:00+00:00                                1```
```

In [ ]:
# In[3]:

# Reuse df_trace, start, end from previous state (they exist)
# 1) compute global P95 per (cmdb_id, trace_name) using full series
trace_p95 = df_trace.groupby(["cmdb_id", "trace_name"])["value"].quantile(0.95).rename("global_p95").reset_index()

# 2) filter to incident window
t_window = df_trace[(df_trace["timestamp"] >= start) & (df_trace["timestamp"] <= end)].copy()

# Prepare summary rows
rows = []
for _, row in trace_p95.iterrows():
    cmdb = row["cmdb_id"]
    trace_name = row["trace_name"]
    thr = row["global_p95"]
    grp = t_window[(t_window["cmdb_id"] == cmdb) & (t_window["trace_name"] == trace_name)].sort_values("timestamp")

    count_in_window = int(grp.shape[0])
    max_in_window = grp["value"].max() if count_in_window > 0 else pd.NA

    if count_in_window > 0:
        b = grp["value"] >= thr
        anomalies_count = int(b.sum())
        earliest_anomaly_timestamp = grp.loc[b, "timestamp"].min() if anomalies_count > 0 else pd.NaT

        if anomalies_count == 0:
            longest_run = 0
        else:
            # compute longest consecutive True run
            bool_series = b.reset_index(drop=True)
            groups = (bool_series != bool_series.shift(1)).cumsum()
            run_df = pd.DataFrame({"grp": groups, "val": bool_series})
            # sum True values within groups, but only consider groups where val is True
            run_lengths = run_df.groupby("grp")["val"].agg(lambda s: int(s.sum()) if s.iloc[0] else 0)
            longest_run = int(run_lengths.max())
    else:
        anomalies_count = 0
        earliest_anomaly_timestamp = pd.NaT
        longest_run = 0

    rows.append({
        "cmdb_id": cmdb,
        "trace_name": trace_name,
        "global_p95": thr,
        "count_in_window": count_in_window,
        "max_in_window": max_in_window,
        "anomalies_count": anomalies_count,
        "earliest_anomaly_timestamp": earliest_anomaly_timestamp,
        "longest_consecutive_anomaly_run": longest_run
    })

trace_anomaly_summary = pd.DataFrame(rows)[[
    "cmdb_id", "trace_name", "global_p95", "count_in_window", "max_in_window",
    "anomalies_count", "earliest_anomaly_timestamp", "longest_consecutive_anomaly_run"
]]

# Sort by anomalies_count desc and limit to top 20
trace_anomaly_summary = trace_anomaly_summary.sort_values(by="anomalies_count", ascending=False).head(20)

# Display compact table
trace_anomaly_summary

```
Out[3]:
```
```python
summary = (
    "Summary of trace anomaly scan (window 2024-01-16T02:43:30Z–2024-01-16T03:13:30Z):\n"
    "- Many trace.error_rate series across components were flagged as 'anomalous' because their global P95 = 0.0.\n"
    "- In every flagged case the in-window max is 0.0, and anomalies_count = 25 with longest run = 25. "
    "This means all points >= 0.0 were counted, but the values are actually zero (no observed errors).\n"
    "- Interpretation: the P95=0.0 threshold makes error_rate traces uninformative here (it trivially flags all points). "
    "There is no evidence of increased error rates in the traces during the incident window.\n"
    "- Recommendation: ignore these trivial error_rate hits, and inspect other trace signals (duration_p95, duration_mean, row_count) "
    "and metrics (we previously found a strong socket anomaly for checkoutservice). Also consider using a non-zero threshold (e.g., >0) "
    "or different anomaly methods for error_rate series."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                 trace_name  global_p95  count_in_window  max_in_window  anomalies_count earliest_anomaly_timestamp  longest_consecutive_anomaly_run
2          checkoutservice      trace.from_checkoutservice.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25
6          checkoutservice      trace.from_frontendservice.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25
14         checkoutservice        trace.to_checkoutservice.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25
10         checkoutservice                 trace.from_root.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25
18         checkoutservice        trace.to_currencyservice.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25
22         checkoutservice           trace.to_emailservice.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25
126                   root        trace.to_checkoutservice.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25
122  recommendationservice  trace.to_recommendationservice.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25
58         frontendservice                 trace.from_root.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25
62         frontendservice        trace.to_checkoutservice.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25
70         frontendservice        trace.to_frontendservice.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25
66         frontendservice        trace.to_currencyservice.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25
82          paymentservice      trace.from_checkoutservice.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25
86          paymentservice                 trace.from_root.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25
78         frontendservice  trace.to_recommendationservice.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25
74         frontendservice  trace.to_productcatalogservice.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25
42         currencyservice                 trace.from_root.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25
46            emailservice      trace.from_checkoutservice.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25
54         frontendservice      trace.from_frontendservice.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25
50            emailservice                 trace.from_root.error_rate         0.0               25            0.0               25  2024-01-16 02:46:00+00:00                               25```
```

In [ ]:
# In[4]:

import pandas as pd

# Load error_logs.csv and parse timestamps as UTC
df_error = pd.read_csv("error_logs.csv")
df_error["timestamp"] = pd.to_datetime(df_error["timestamp"], unit="s", utc=True)

# Ensure incident window variables exist (reuse from prior cells)
# start, end were defined earlier; if not, define them again
try:
    start, end
except NameError:
    start = pd.to_datetime("2024-01-16T02:43:30Z", utc=True)
    end = pd.to_datetime("2024-01-16T03:13:30Z", utc=True)

# Filter to incident window
err_win = df_error[(df_error["timestamp"] >= start) & (df_error["timestamp"] <= end)].copy()

# Prepare message lowercase for case-insensitive search
err_win["message_lower"] = err_win["message"].fillna("").str.lower()

# Keywords to search (case-insensitive substrings)
keywords = ['socket', 'connection', 'timeout', 'refused', 'reset', 'closed', 'pool', 'exhaust']

rows = []
for kw in keywords:
    mask = err_win["message_lower"].str.contains(kw, na=False)
    if mask.any():
        matched = err_win[mask]
        grp = matched.groupby("cmdb_id").agg(
            match_count=("message", "count"),
            earliest_match_timestamp=("timestamp", "min")
        ).reset_index()
        grp["keyword"] = kw
        # reorder columns
        grp = grp[["cmdb_id", "keyword", "match_count", "earliest_match_timestamp"]]
        rows.append(grp)

if rows:
    error_keyword_summary = pd.concat(rows, ignore_index=True)
    # sort and limit top 20
    error_keyword_summary = error_keyword_summary.sort_values(by="match_count", ascending=False).head(20)
else:
    # One-row note as requested when zero matches overall
    error_keyword_summary = pd.DataFrame([{
        "note": "no keyword matches in error_logs in-window"
    }])

error_keyword_summary

```
Out[4]:
```
```python
summary = (
    "During the incident window 2024-01-16T02:43:30Z–2024-01-16T03:13:30Z, a case-insensitive substring search "
    "for ['socket','connection','timeout','refused','reset','closed','pool','exhaust'] in error_logs.csv "
    "returned no matches. In other words, there is no direct error_log evidence (no matched messages) pointing to "
    "socket/connection/pool/timeout issues in that time range. Corroborate this with the metric/trace findings (e.g., "
    "checkoutservice socket anomalies) when prioritizing investigation."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

note
0  no keyword matches in error_logs in-window```
```